# 掃描綱要 PDF → OCR → Markdown（每篇一段＋檢查報告）

範例 `ex_TS26TRN_zht.pdf` 87 頁都是圖片、沒有文字層，所以要 OCR。這本筆記本用 **Google 雲端硬碟內建的 OCR**（免費、用你自己的帳號、不用 GPU）：
每頁轉成圖片 → 暫時建立 Google 文件做 OCR → 取出文字 → 刪除暫存文件。87 頁約 5–10 分鐘。

會要求兩次授權：連接雲端硬碟（讀 PDF、寫結果），以及 Google 帳號（呼叫 Drive OCR）。都按允許。

輸出在 PDF 旁邊：
- `<檔名>-ocr/`：每頁 OCR 文字（快取；重跑只補缺的頁）。
- `<檔名>-md/<檔名>.md`、`<檔名>-md/<檔名>/NN.md`（每篇一檔）、`<檔名>.report.json`（含自動檢查：不存在的章數、和專名只差一字的詞）。

已經 OCR 過的頁有快取，重跑只需 1–2 分鐘（不會重新 OCR）。

**準確度**：這份是「綱要附經文」版，筆記本會拿 OCR 出來的經文和雲端硬碟上的恢復本（`twgbr-archive/recovery-version/text`）逐字比對，算出 OCR 錯字率。

跑完回覆「直接讀」。

In [ ]:
# 第 1 格：設定
PDF_NAME = 'ex_TS26TRN_zht.pdf'   # 雲端硬碟上的綱要 PDF 檔名（自動搜尋 MyDrive 前 4 層資料夾）

In [ ]:
# 第 2 格：連接雲端硬碟、授權 Drive OCR、找到 PDF
import os, subprocess, sys, json
from pathlib import Path
from google.colab import auth, drive
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
SERVICE = build('drive', 'v3')
ROOT = Path('/content/drive/MyDrive')
found = [Path(d)/PDF_NAME for d, _, files in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) <= 4 and PDF_NAME in files]
if not found:
    raise RuntimeError(f'找不到 {PDF_NAME}；請確認檔名，或把 PDF 放在 MyDrive 前 4 層資料夾內')
PDF = found[0]
archives = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'recovery-version' in dirs]
RV_TEXT = archives[0]/'recovery-version'/'text' if archives and (archives[0]/'recovery-version'/'text').is_dir() else None
print('恢復本經文：', RV_TEXT or '找不到（只做章數檢查，不算 OCR 錯字率）')
print('PDF：', PDF, f'（{PDF.stat().st_size/1e6:.1f} MB）')

In [ ]:
# 第 3 格：取得固定版本的程式與套件
COMMIT = '5eb91f610e1b846a14a7262a179fd25dca385313'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymupdf==1.26.5'], check=True)
sys.path.insert(0, str(REPO/'scripts'))
import importlib, outline_pdf_to_md, outline_ocr
importlib.reload(outline_pdf_to_md); importlib.reload(outline_ocr)
print('程式版本', COMMIT[:12])

In [ ]:
# 第 4 格：OCR（已有快取的頁跳過）並轉成 Markdown，用恢復本檢查準確度
report = outline_ocr.run(PDF, PDF.parent/(PDF.stem + '-ocr'), PDF.parent/(PDF.stem + '-md'), SERVICE,
                         log=lambda message: print(message, flush=True), recovery_version_dir=RV_TEXT)
print('\n頁數', report['pages'], '｜警告：', *report['warnings'], sep='\n')
for m in report['messages']:
    print(f"第{m['number']}篇 {m['title'][:24]}：讀經 {m['scripture_lines']} 行、要點 {m['points']}、未標號行 {m['unmarked_lines']}、附經文 {m['quoted_verses']} 節")
check = report.get('verse_check')
if check:
    print(f"\nOCR 經文比對：{check['quotes_checked']} 節、{check['ocr_chars']} 字、錯 {check['errors']} 字 → 錯字率 {check['cer']:.2%}"
          f"（印刷版與網站用字不同也算錯時 {check['cer_counting_print_variants']:.2%}）")
    for w in check['worst'][:10]:
        print(f"  {w['ref']}（{w['errors']} 字）OCR：{w['ocr'][:40]}\n{'':>10}恢復本：{w['recovery_version'][:40]}")
print('\n結果：', PDF.parent/(PDF.stem + '-md'))